# Preprocessing & Train/Val/Test Split

Apply the preprocessing pipeline and create stratified splits
for model training and evaluation.

In [2]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))

import pandas as pd
from sklearn.model_selection import train_test_split

from config import (
    REVIEWS_RAW, TRAIN_CSV, VAL_CSV, TEST_CSV,
    TARGET_COL, MIN_REVIEW_LENGTH, RANDOM_STATE,
    TEST_SIZE, VAL_SIZE, DATA_PROCESSED
)
from src.preprocessing import preprocess_dataframe

# Make sure output directory exists
DATA_PROCESSED.mkdir(parents=True, exist_ok=True)

In [3]:
# Load raw data
df = pd.read_csv(REVIEWS_RAW)
print(f"Raw dataset: {len(df)} rows")

# Apply preprocessing
df = preprocess_dataframe(df, min_length=MIN_REVIEW_LENGTH)
print(f"After preprocessing: {len(df)} rows")
print(f"Removed: {10000 - len(df)} rows")
print(f"\nClass distribution after preprocessing:")
print(df[TARGET_COL].value_counts())

Raw dataset: 10000 rows
After preprocessing: 8896 rows
Removed: 1104 rows

Class distribution after preprocessing:
recommendation
Recommended        7089
Not Recommended    1807
Name: count, dtype: int64


In [4]:
# Stratified train/val/test split
# Step 1: Split off test set (15%)
train_val, test = train_test_split(
    df, test_size=TEST_SIZE, random_state=RANDOM_STATE,
    stratify=df[TARGET_COL]
)

# Step 2: Split train_val into train and val (15% of original = ~17.6% of train_val)
val_fraction = VAL_SIZE / (1 - TEST_SIZE)
train, val = train_test_split(
    train_val, test_size=val_fraction, random_state=RANDOM_STATE,
    stratify=train_val[TARGET_COL]
)

print(f"Train: {len(train)} rows ({len(train)/len(df)*100:.1f}%)")
print(f"Val:   {len(val)} rows ({len(val)/len(df)*100:.1f}%)")
print(f"Test:  {len(test)} rows ({len(test)/len(df)*100:.1f}%)")

print(f"\nClass balance check:")
for name, split in [("Train", train), ("Val", val), ("Test", test)]:
    pct = split[TARGET_COL].value_counts(normalize=True) * 100
    print(f"  {name}: {pct['Recommended']:.1f}% Rec / {pct['Not Recommended']:.1f}% Not Rec")

Train: 6226 rows (70.0%)
Val:   1335 rows (15.0%)
Test:  1335 rows (15.0%)

Class balance check:
  Train: 79.7% Rec / 20.3% Not Rec
  Val: 79.7% Rec / 20.3% Not Rec
  Test: 79.7% Rec / 20.3% Not Rec


In [5]:
# Save processed splits
train.to_csv(TRAIN_CSV, index=False)
val.to_csv(VAL_CSV, index=False)
test.to_csv(TEST_CSV, index=False)

print(f"Saved:")
print(f"  {TRAIN_CSV} ({len(train)} rows)")
print(f"  {VAL_CSV} ({len(val)} rows)")
print(f"  {TEST_CSV} ({len(test)} rows)")

Saved:
  C:\Users\user\steam-reviews-nlp\data\processed\train.csv (6226 rows)
  C:\Users\user\steam-reviews-nlp\data\processed\val.csv (1335 rows)
  C:\Users\user\steam-reviews-nlp\data\processed\test.csv (1335 rows)
